# Merge Light Curve Files with MJD from DP2 Visit Table

This notebook enriches every light-curve file produced by
`01_MatchTargetsWithLSSTCamSources.ipynb` with the precise observation
mid-point time (`expMidptMJD`) from the official DP2 visit table.

## Strategy

1. Load the DP2 visit table (`dp2_visits_table_with_iq.ecsv`) and build a
   lookup dictionary keyed on `visitId`.
2. For each CSV/Parquet light-curve file in `data_MATCHSRC_01_out/` (both the
   global concatenated file and the per-star files), merge on the `visit`
   column to attach `expMidptMJD`, `obsStartMJD`, `band_visit`, `airmass`,
   `mean_seeing`, and `mean_maglim`.
3. Write the enriched files to `data_MergeVisits_02_out/` mirroring the same
   sub-directory layout (`per_star/`).

The column rename `band → band_visit` for the visit-table band avoids a name
collision with the `band` column already present in the light-curve files.

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-27
- **Last update:** 2026-06-27


## 1. Imports

In [ ]:
import gc
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from astropy.io import ascii as astropy_ascii

## 2. Logging setup

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Logging initialised.")

## 3. Configuration — NB_TAG, directories, parameters

In [ ]:
# ── Notebook tag used as prefix for output directories / figure names ──────
NB_TAG = "MergeVisits_02"

# ── Input: visit table ──────────────────────────────────────────────────────
DIR_VISITS_IN = "./data_MergeVisits_02_in"
VISITS_FILE = "dp2_visits_table_with_iq.ecsv"

# ── Input: light curves produced by notebook 01 ─────────────────────────────
DIR_LC_IN = "./data_MATCHSRC_01_out"
DIR_LC_PER_STAR_IN = os.path.join(DIR_LC_IN, "per_star")

# ── Output ───────────────────────────────────────────────────────────────────
DIR_DATA = f"./data_{NB_TAG}_out"
DIR_DATA_PER_STAR = os.path.join(DIR_DATA, "per_star")
DIR_FIGS = f"./figs_{NB_TAG}"

for d in [DIR_DATA, DIR_DATA_PER_STAR, DIR_FIGS]:
    os.makedirs(d, exist_ok=True)
    log.info("Directory ready: %s", d)

# ── Columns imported from the visit table ────────────────────────────────────
# Rename 'band' from the visit table to avoid collision with 'band' in LCs.
VISIT_COLS_KEEP = [
    "visitId",
    "expMidptMJD",
    "obsStartMJD",
    "band",  # will be renamed to band_visit
    "airmass",
    "mean_seeing",
    "mean_maglim",
    "expTime",
    "zenithDistance",
]

# Columns inserted right after 'visit' in the merged output
MJD_COL = "expMidptMJD"
T0_COL = "obsStartMJD"

## 4. Load the DP2 visit table and build the lookup

In [ ]:
visits_path = os.path.join(DIR_VISITS_IN, VISITS_FILE)
log.info("Reading visit table: %s", visits_path)

# astropy ascii handles the ECSV format (including units/meta)
visits_tbl = astropy_ascii.read(visits_path)
log.info("Visit table: %d rows, columns: %s", len(visits_tbl), visits_tbl.colnames)

# Convert to pandas for easy merging
df_visits = visits_tbl.to_pandas()
log.info("Converted to DataFrame. Shape: %s", df_visits.shape)

In [ ]:
# Preview the visit table
df_visits.head(3)

In [ ]:
# Keep only the columns we will attach to light curves
existing_visit_cols = [c for c in VISIT_COLS_KEEP if c in df_visits.columns]
missing_visit_cols = [c for c in VISIT_COLS_KEEP if c not in df_visits.columns]
if missing_visit_cols:
    log.warning("Visit table columns NOT found (skipped): %s", missing_visit_cols)

df_visits_slim = df_visits[existing_visit_cols].copy()

# Rename 'band' to 'band_visit' to avoid collision with 'band' in the LC files
if "band" in df_visits_slim.columns:
    df_visits_slim = df_visits_slim.rename(columns={"band": "band_visit"})

# Ensure visitId is an integer (same dtype as 'visit' in the LC files)
df_visits_slim["visitId"] = df_visits_slim["visitId"].astype(np.int64)

log.info("Visit lookup table: %d rows, columns: %s", len(df_visits_slim), df_visits_slim.columns.tolist())
df_visits_slim.head(3)

## 5. Helper function: merge one LC DataFrame with the visit lookup

In [ ]:
def merge_lc_with_visits(
    df_lc: pd.DataFrame,
    df_visit_lookup: pd.DataFrame,
) -> pd.DataFrame:
    """Merge a light-curve DataFrame with the visit lookup table.

    Parameters
    ----------
    df_lc:
        Light-curve DataFrame that contains a 'visit' column (int64).
    df_visit_lookup:
        Slimmed visit table with 'visitId' as key and MJD / quality columns.

    Returns
    -------
    Merged DataFrame with MJD columns inserted right after 'visit'.
    Rows without a matching visitId are kept but their MJD columns are NaN.
    """
    df_lc = df_lc.copy()
    # Ensure compatible key type
    df_lc["visit"] = df_lc["visit"].astype(np.int64)

    df_merged = df_lc.merge(
        df_visit_lookup,
        left_on="visit",
        right_on="visitId",
        how="left",
    )

    # Drop the redundant 'visitId' column (same value as 'visit')
    if "visitId" in df_merged.columns:
        df_merged = df_merged.drop(columns=["visitId"])

    # Re-order columns: insert MJD columns right after 'visit'
    new_cols = [c for c in df_visit_lookup.columns if c != "visitId"]
    orig_cols = df_lc.columns.tolist()
    visit_idx = orig_cols.index("visit") + 1
    ordered = orig_cols[:visit_idx] + new_cols + orig_cols[visit_idx:]
    # Keep only columns that ended up in the merged DataFrame
    ordered = [c for c in ordered if c in df_merged.columns]
    df_merged = df_merged[ordered]

    n_missing = df_merged[MJD_COL].isna().sum() if MJD_COL in df_merged.columns else 0
    if n_missing:
        log.warning("  %d rows have no matching visitId (MJD = NaN)", n_missing)

    return df_merged

## 6. Helper function: save merged DataFrame (CSV + Parquet)

In [ ]:
def save_merged(
    df: pd.DataFrame,
    out_dir: str,
    basename: str,
) -> None:
    """Save *df* as both CSV and Parquet under *out_dir*.

    Parameters
    ----------
    df        : merged DataFrame to save.
    out_dir   : output directory (must already exist).
    basename  : file stem, e.g. 'all_stars_lightcurves_mjd'.
    """
    csv_path = os.path.join(out_dir, basename + ".csv")
    parquet_path = os.path.join(out_dir, basename + ".parquet")
    df.to_csv(csv_path, index=False)
    df.to_parquet(parquet_path, index=False)
    log.info("  Saved  CSV    : %s  (%d rows)", csv_path, len(df))
    log.info("  Saved  Parquet: %s", parquet_path)

## 7. Merge and save the global concatenated light-curve file

In [ ]:
# ── Load global LC file ──────────────────────────────────────────────────────
all_lc_csv = os.path.join(DIR_LC_IN, "all_stars_lightcurves.csv")
log.info("Loading global LC file: %s", all_lc_csv)
df_all = pd.read_csv(all_lc_csv)
log.info("  Shape before merge: %s", df_all.shape)

# ── Merge ────────────────────────────────────────────────────────────────────
df_all_merged = merge_lc_with_visits(df_all, df_visits_slim)
log.info("  Shape after  merge: %s", df_all_merged.shape)

# ── Save ─────────────────────────────────────────────────────────────────────
save_merged(df_all_merged, DIR_DATA, "all_stars_lightcurves_mjd")

In [ ]:
# Quick sanity check
print(f"MJD range: {df_all_merged[MJD_COL].min():.4f}  –  {df_all_merged[MJD_COL].max():.4f}")
print(f"NaN MJD  : {df_all_merged[MJD_COL].isna().sum()} / {len(df_all_merged)}")
df_all_merged[["simbad_id", "visit", MJD_COL, T0_COL, "band", "band_visit"]].head(5)

## 8. Also copy the summary file (no merge needed, just copy)

In [ ]:
import shutil

summary_src = os.path.join(DIR_LC_IN, "lightcurve_match_summary.csv")
summary_dst = os.path.join(DIR_DATA, "lightcurve_match_summary.csv")

if os.path.exists(summary_src):
    shutil.copy2(summary_src, summary_dst)
    log.info("Copied summary file to: %s", summary_dst)
else:
    log.warning("Summary file not found: %s", summary_src)

## 9. Merge and save per-star light-curve files

In [ ]:
# Discover all per-star CSV files (skip Parquet — we regenerate them)
per_star_csv_files = sorted(f for f in os.listdir(DIR_LC_PER_STAR_IN) if f.endswith("_lc.csv"))
log.info("Found %d per-star CSV files in %s", len(per_star_csv_files), DIR_LC_PER_STAR_IN)
per_star_csv_files[:5]

In [ ]:
n_ok = 0
n_err = 0

for fname in per_star_csv_files:
    src_path = os.path.join(DIR_LC_PER_STAR_IN, fname)

    try:
        df_star = pd.read_csv(src_path)
    except Exception as exc:
        log.error("  ERROR reading %s : %s", fname, exc)
        n_err += 1
        continue

    if "visit" not in df_star.columns:
        log.warning("  SKIP %s — no 'visit' column found.", fname)
        n_err += 1
        continue

    # Merge with visit lookup
    df_star_merged = merge_lc_with_visits(df_star, df_visits_slim)

    # Output basename keeps the original stem, with '_mjd' suffix
    stem = fname.replace("_lc.csv", "")
    out_stem = stem + "_lc_mjd"
    save_merged(df_star_merged, DIR_DATA_PER_STAR, out_stem)

    n_ok += 1
    gc.collect()

log.info("Per-star merge done: %d OK, %d errors.", n_ok, n_err)

## 10. Diagnostic: MJD coverage overview

In [ ]:
# Summary statistics of MJD coverage per band across the full merged table
if MJD_COL in df_all_merged.columns and "band" in df_all_merged.columns:
    stats = (
        df_all_merged.dropna(subset=[MJD_COL])
        .groupby("band")[MJD_COL]
        .agg(["count", "min", "max"])
        .rename(columns={"count": "n_visits", "min": "MJD_min", "max": "MJD_max"})
    )
    stats["MJD_span_days"] = stats["MJD_max"] - stats["MJD_min"]
    print("MJD coverage per band:")
    print(stats.to_string())

## 11. Quick-look plots: example light curve with MJD time axis

In [ ]:
# Pick the first per-star merged file as a demo
demo_files = sorted(f for f in os.listdir(DIR_DATA_PER_STAR) if f.endswith("_lc_mjd.csv"))

if not demo_files:
    log.warning("No merged per-star file found for demo plot.")
else:
    demo_path = os.path.join(DIR_DATA_PER_STAR, demo_files[0])
    df_demo = pd.read_csv(demo_path)

    simbad_id_demo = df_demo["simbad_id"].iloc[0] if "simbad_id" in df_demo.columns else "unknown"
    bands_demo = df_demo["band"].unique() if "band" in df_demo.columns else []

    fig, ax = plt.subplots(figsize=(10, 4))

    cmap = plt.get_cmap("tab10")
    band_colors = {b: cmap(i) for i, b in enumerate(sorted(bands_demo))}

    for band, grp in df_demo.dropna(subset=[MJD_COL]).groupby("band"):
        if "psfFlux" not in grp.columns:
            continue
        ax.errorbar(
            grp[MJD_COL],
            grp["psfFlux"],
            yerr=grp.get("psfFluxErr", None),
            fmt="o",
            ms=4,
            label=band,
            color=band_colors.get(band),
            alpha=0.75,
        )

    ax.set_xlabel("expMidptMJD")
    ax.set_ylabel("psfFlux  [nJy]")
    ax.set_title(f"Light curve (demo) — {simbad_id_demo}")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()

    figname = f"lc_mjd_demo_{simbad_id_demo.replace(' ', '_')}"

    def savefig(fig, name, dpi=150):
        """Save figure as both PDF and PNG."""
        fig.savefig(os.path.join(DIR_FIGS, name + ".pdf"), dpi=dpi, bbox_inches="tight")
        fig.savefig(os.path.join(DIR_FIGS, name + ".png"), dpi=dpi, bbox_inches="tight")
        log.info("Figure saved: %s (.pdf/.png)", os.path.join(DIR_FIGS, name))

    savefig(fig, figname)
    plt.show()

## 12. Output directory listing

In [ ]:
print(f"\n=== Contents of {DIR_DATA} ===")
for entry in sorted(os.listdir(DIR_DATA)):
    full = os.path.join(DIR_DATA, entry)
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print(f"  [DIR]  {entry}/  ({n} files)")
    else:
        size_kb = os.path.getsize(full) / 1024
        print(f"  [FILE] {entry}  ({size_kb:.1f} kB)")